In [ ]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================================
# pandas: Manipulación y análisis de datos
# openmeteo_requests: Cliente para acceder a la API de Open-Meteo
# requests_cache: Cachea las solicitudes HTTP para evitar duplicados
# retry_requests: Reintenta solicitudes fallidas automáticamente

import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [ ]:
# ============================================================================
# CARGAR DATOS HISTÓRICOS
# ============================================================================
# Lee el archivo parquet unificado de datos climáticos
# Este archivo contiene datos de múltiples estaciones meteorológicas

df = pd.read_parquet("../local_cleanup/clima_unificado.parquet")
df.head()

,station_id,obs_timestamp,temp_real,dew_real,hum_real,wind_speed_real,wind_gust_real,wind_dir_real,press_real,sea_level_press_real,...,fcst_start,fcst_end,is_daytime_fcst,temp_fcst,dew_fcst,hum_fcst,wind_speed_fcst,wind_dir_fcst,precip_prob_fcst,short_fcst
0,KXBP,2026-04-16 04:10:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2026-04-16 04:00:00+00:00,2026-04-16 05:00:00+00:00,False,20.000000,18.333333,90.0,10 mph,S,5.0,Partly Cloudy
1,KPVW,2026-04-16 04:10:00+00:00,11.9,-11.6,18.157693,0.00,NaN,0.0,101360.0,NaN,...,2026-04-16 04:00:00+00:00,2026-04-16 05:00:00+00:00,False,15.000000,-7.222222,21.0,5 mph,WSW,0.0,Clear
2,KCQB,2026-04-16 04:10:00+00:00,19.0,17.4,90.445091,20.52,NaN,190.0,101080.0,NaN,...,2026-04-16 04:00:00+00:00,2026-04-16 05:00:00+00:00,False,18.888889,17.222222,90.0,9 mph,S,11.0,Partly Cloudy
3,KVYS,2026-04-16 04:10:00+00:00,15.0,14.1,94.355349,20.52,NaN,30.0,100950.0,NaN,...,2026-04-16 04:00:00+00:00,2026-04-16 05:00:00+00:00,False,18.888889,16.111111,85.0,15 mph,SSW,76.0,Showers And Thunderstorms
4,KOJA,2026-04-16 04:10:00+00:00,18.9,15.9,82.740233,12.96,NaN,210.0,101080.0,NaN,...,2026-04-16 04:00:00+00:00,2026-04-16 05:00:00+00:00,False,17.777778,8.333333,54.0,7 mph,S,3.0,Mostly Clear


In [ ]:
# ============================================================================
# EXPLORAR ESTRUCTURA DE DATOS
# ============================================================================
# Muestra todas las columnas disponibles en el dataset

df.columns

Index(['station_id', 'obs_timestamp', 'temp_real', 'dew_real', 'hum_real',
       'wind_speed_real', 'wind_gust_real', 'wind_dir_real', 'press_real',
       'sea_level_press_real', 'precip_1h_real', 'visibility_real', 'zona_id',
       'lat_estacion', 'lon_estacion', 'estado', 'fcst_start', 'fcst_end',
       'is_daytime_fcst', 'temp_fcst', 'dew_fcst', 'hum_fcst',
       'wind_speed_fcst', 'wind_dir_fcst', 'precip_prob_fcst', 'short_fcst'],
      dtype='str')

In [ ]:
# ============================================================================
# OBTENER ESTACIONES ÚNICAS
# ============================================================================
# Extrae todos los identificadores de estaciones (station_id) únicos del dataset
# Esto identifica cuántas y cuáles estaciones meteorológicas hay

estaciones = list(df["station_id"].unique())
print(estaciones)

['KXBP', 'KPVW', 'KCQB', 'KVYS', 'KOJA', 'KOFK', 'KXVG', 'KETH', 'KJSO', 'KRCR', 'KFKR', 'K4O4', 'KDNV', 'KDCY', 'KI67', 'KCBF', 'KMZZ', 'KTYQ', 'KMLE', 'KMQJ', 'KGPC', 'KRZL', 'KF05', 'KPSN', 'KIBM', 'KJWG', 'KADH', 'KT89', 'KADM', 'KOKM', 'KVHN', 'KE38', 'KAVK', 'KSOA', 'KBAX', 'KPEQ', 'KBXA', 'KHZR', 'KLUV', 'K9D7', 'KMKN', 'KBTA', 'KD95', 'KMDD', 'KGNC', 'KDUA', 'KHDC', 'KPMV', 'KCIN', 'KAHQ', 'KHHF', 'KPPA', 'KHRX', 'KIKV', 'KTNU', 'KPRO', 'KBNW', 'KPEA', 'KI75', 'KEHA', 'KEBS', 'KOOA', 'KIFA', 'KTVK', 'KCAV', 'KAWG', 'KIIB', 'KAXA', 'KPTT', 'KOLZ', 'KY19', 'K5H4', 'KBWW', 'KBAC', 'KS32', 'KGWR', 'K2D5', 'KCCA', 'KFET', 'KWWR', 'K9V9', 'KIER', 'KHNR', 'KRDK', 'KSDA', 'KAFK', 'KAIO', 'KDNS', 'KICL', 'KADU', 'KATA', 'KCSQ', 'KLCG', 'KJYR', 'KSLB', 'KMYZ', 'KLRJ', 'KMNE', 'KJKJ', 'KDTL', 'KCKN', 'KSHL', 'KADC', 'KCNB', 'KPQN', 'K1D8', 'KLXL', 'KROX', 'K98D', 'KCHK', 'KE11', 'K06D', 'KLUD', 'K21D', 'KCFE', 'KOEO', 'KSYN', 'KRNH', 'KGVT', 'KGYL', 'KSGS', 'KEFT', 'KPNM', 'KDLL', 'KF46',

In [ ]:
# ============================================================================
# OBTENER COORDENADAS DE ESTACIONES
# ============================================================================
# Extrae las coordenadas geográficas (latitud y longitud) únicas de cada estación
# Elimina duplicados ya que cada estación tiene una ubicación fija

estaciones_coords = df[['station_id', 'lat_estacion', 'lon_estacion']].drop_duplicates().reset_index(drop=True)
print(f"Número de estaciones únicas: {len(estaciones_coords)}")
estaciones_coords.head()

Número de estaciones únicas: 520


,station_id,lat_estacion,lon_estacion
0,KXBP,33.17528,-97.82833
1,KPVW,34.16806,-101.71722
2,KCQB,35.72389,-96.82028
3,KVYS,41.35175,-89.14963
4,KOJA,35.54472,-98.66833


In [ ]:
# ============================================================================
# CONFIGURACIÓN: PERÍODO Y VARIABLES
# ============================================================================

# PERÍODO TEMPORAL
# Define el rango de fechas para obtener datos históricos
start_date = "2025-10-01"  # Fecha inicial (1 de octubre de 2025)
end_date = "2026-04-15"    # Fecha final (15 de abril de 2026)

# VARIABLES HORARIAS DE LA API
# Lista de variables meteorológicas a descargar de Open-Meteo
hourly_vars = [
    "temperature_2m",        # Temperatura a 2 metros
    "dew_point_2m",           # Punto de rocío
    "relative_humidity_2m",  # Humedad relativa
    "precipitation",         # Precipitación
    "pressure_msl",          # Presión a nivel del mar
    "surface_pressure",      # Presión de superficie
    "wind_speed_10m",         # Velocidad del viento a 10 metros
    "wind_direction_10m",     # Dirección del viento
    "wind_gusts_10m",         # Ráfagas de viento
    "visibility",            # Visibilidad
    "is_day"                 # Indicador de día/noche
]

# MAPEO DE NOMBRES
# Convierte nombres de variables de la API a nombres del dataset local
var_mapping = {
    "temperature_2m": "temp",
    "dew_point_2m": "dew",
    "relative_humidity_2m": "hum",
    "precipitation": "precip_1h",
    "pressure_msl": "sea_level_press",
    "surface_pressure": "press",
    "wind_speed_10m": "wind_speed",
    "wind_direction_10m": "wind_dir",
    "wind_gusts_10m": "wind_gust",
    "visibility": "visibility",
    "is_day": "is_daytime"
}

# MODELO DE PRONÓSTICO
# Especifica qué modelo usar para pronósticos históricos
model = ["gfs_seamless"]

In [ ]:
# ============================================================================
# DESCARGAR Y PROCESAR DATOS HISTÓRICOS
# ============================================================================
# Esta celda:
# 1. Configura la sesión con caché y reintentos
# 2. Para cada estación, obtiene observaciones y pronósticos históricos
# 3. Combina ambos en un único dataframe

# CONFIGURACIÓN DE CLIENTE API
# - CachedSession: Cachea resultados para evitar solicitudes duplicadas
# - retry_session: Reintenta automáticamente si falla la solicitud
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Lista para almacenar todos los dataframes de estaciones
historicos_list = []

# ============================================================================
# ITERAR POR CADA ESTACIÓN
# ============================================================================
for idx, row in estaciones_coords.iterrows():
    station_id = row['station_id']
    lat = row['lat_estacion']
    lon = row['lon_estacion']
    
    print(f"Procesando estación {station_id} en ({lat}, {lon})")
    
    # ========================================================================
    # OBTENER OBSERVACIONES HISTÓRICAS
    # ========================================================================
    # Descarga datos reales observados en esas fechas y coordenadas
    try:
        url_obs = "https://archive-api.open-meteo.com/v1/archive"
        params_obs = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": hourly_vars
        }
        responses_obs = openmeteo.weather_api(url_obs, params_obs)
        response_obs = responses_obs[0]
        
        # Procesar datos horarios
        hourly_obs = response_obs.Hourly()
        times = pd.date_range(
            start=pd.to_datetime(hourly_obs.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly_obs.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly_obs.Interval()),
            inclusive="left"
        )
        
        # Construir dataframe con observaciones
        data_obs = {"time": times, "station_id": station_id}
        for i, var in enumerate(hourly_vars):
            data_obs[f"{var_mapping[var]}_observation"] = hourly_obs.Variables(i).ValuesAsNumpy()
        
        df_obs = pd.DataFrame(data_obs)
        
    except Exception as e:
        print(f"Error obteniendo observaciones para {station_id}: {e}")
        continue
    
    # ========================================================================
    # OBTENER PRONÓSTICOS HISTÓRICOS
    # ========================================================================
    # Descarga los pronósticos que se hicieron para esas mismas fechas
    # Permite comparar predicciones vs realidad
    try:
        url_fcst = "https://historical-forecast-api.open-meteo.com/v1/forecast"
        params_fcst = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": hourly_vars,
            "models": model
        }
        responses_fcst = openmeteo.weather_api(url_fcst, params_fcst)
        response_fcst = responses_fcst[0]
        
        # Procesar datos horarios de pronóstico
        hourly_fcst = response_fcst.Hourly()
        times_fcst = pd.date_range(
            start=pd.to_datetime(hourly_fcst.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly_fcst.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly_fcst.Interval()),
            inclusive="left"
        )
        
        # Construir dataframe con pronósticos
        data_fcst = {"time": times_fcst, "station_id": station_id}
        for i, var in enumerate(hourly_vars):
            data_fcst[f"{var_mapping[var]}_prediction"] = hourly_fcst.Variables(i).ValuesAsNumpy()
        
        df_fcst = pd.DataFrame(data_fcst)
        
    except Exception as e:
        print(f"Error obteniendo pronósticos para {station_id}: {e}")
        continue
    
    # ========================================================================
    # COMBINAR OBSERVACIONES Y PRONÓSTICOS
    # ========================================================================
    # Une los dos dataframes por timestamp y station_id
    # Resultado: para cada hora tenemos valor real y valor predicho
    df_hist = pd.merge(df_obs, df_fcst, on=["time", "station_id"], how="outer")
    historicos_list.append(df_hist)

# ============================================================================
# CONSOLIDAR RESULTADOS
# ============================================================================
# Concatena todos los dataframes de estaciones en uno solo
if historicos_list:
    df_historicos = pd.concat(historicos_list, ignore_index=True)
    print(f"Dataset histórico creado con {len(df_historicos)} filas")
    df_historicos.head()
else:
    print("No se obtuvieron datos históricos")

In [ ]:
# ============================================================================
# GUARDAR DATASET HISTÓRICO
# ============================================================================
# Exporta el dataframe consolidado a formato Parquet
# Parquet es eficiente para grandes volúmenes de datos

if 'df_historicos' in locals():
    df_historicos.to_parquet("historicos.parquet", index=False)
    print("Dataset histórico guardado en 'historicos.parquet'")
else:
    print("No hay df_historicos para guardar")

Dataset histórico guardado en 'historicos.parquet'


In [ ]:
#Falta poner la zona_id en historicos.parquet